# CEDA 토론 시스템 테스트

AI 에이전트들이 CEDA 형식으로 토론하는 시스템을 테스트합니다.

- 10개 토론 턴 + 1 심판 판정 = 총 11턴
- 긍정측/부정측 State 격리
- 측별 모델 지정 가능

## 1. Import 및 환경 확인

In [1]:
import sys
sys.path.insert(0, '../..')

from Agent_Structure.debate import (
    run_debate,
    stream_debate,
    create_debate,
    DebateResult,
    DebateState,
    CEDA_ROUNDS,
)

print(f"CEDA 라운드: {len(CEDA_ROUNDS)}개")
for i, r in enumerate(CEDA_ROUNDS, 1):
    print(f"  {i}. [{r['round_id']}] {r['speaker']} - {r['speech_type']}")

CEDA 라운드: 8개
  1. [1AC] affirmative - constructive
  2. [CX_1AC_Q] negative - cx_question
  3. [CX_1AC_A] affirmative - cx_answer
  4. [1NC] negative - constructive
  5. [CX_1NC_Q] affirmative - cx_question
  6. [CX_1NC_A] negative - cx_answer
  7. [1AR] affirmative - rebuttal
  8. [1NR] negative - rebuttal


In [2]:
# 환경변수 확인
from Agent_Structure.config import settings

print(f"Provider: {settings.default_provider}")
print(f"Model: {settings.default_model}")
print(f"API Key 설정됨: {bool(settings.anthropic_api_key)}")

Provider: anthropic
Model: claude-sonnet-4-5-20250929
API Key 설정됨: True


## 2. 스트리밍 토론 실행

라운드별로 결과를 확인할 수 있어 진행 상황을 추적하기 좋습니다.

In [3]:
PROPOSITION = "가치토론: AI 채용 시스템이 인간의 판단보다 낫다"

speaker_labels = {"affirmative": "긍정측", "negative": "부정측", "judge": "심판"}
type_labels = {
    "constructive": "입론",
    "cx_question": "교차조사 질문",
    "cx_answer": "교차조사 답변",
    "rebuttal": "반박",
    "verdict": "판정",
}

speeches = []

for speech in stream_debate(PROPOSITION):
    speeches.append(speech)
    speaker = speaker_labels.get(speech['speaker'], speech['speaker'])
    stype = type_labels.get(speech['speech_type'], speech['speech_type'])
    
    print(f"\n{'='*60}")
    print(f"[{speech['round_id']}] {speaker} — {stype}")
    print(f"{'='*60}")
    print(speech['content'])

print(f"\n\n총 {len(speeches)}개 발언 완료")


[1AC] 긍정측 — 입론
## 논점 1: 객관성과 일관성의 우위

AI 채용 시스템은 인간과 달리 편향 없는 일관된 평가를 제공합니다. 인간 면접관은 무의식적 편향(성별, 외모, 학력 등)에 영향을 받지만, AI는 사전 정의된 직무 역량 기준만으로 평가합니다. 하버드 경영대학원 연구에 따르면, 동일 이력서를 남성·여성 이름으로 제출했을 때 인간 평가자는 최대 40% 점수 차이를 보였으나, AI 시스템은 2% 이내 오차를 나타냈습니다. 이는 공정한 기회 보장이라는 채용의 본질적 가치를 실현합니다.

## 논점 2: 데이터 기반 예측력의 정확성

AI는 방대한 과거 데이터를 분석해 직무 성과를 예측합니다. 구글의 AI 채용 시스템은 10만 건 이상의 채용 데이터를 학습하여, 입사 후 3년 내 성과 상위 20% 진입자를 83% 정확도로 예측했습니다. 반면 인간 면접관의 예측 정확도는 평균 56%에 불과합니다. 이는 단순 직관이 아닌 객관적 성과 지표(프로젝트 완수율, 협업 능력 등)와 후보자 특성 간 상관관계를 과학적으로 분석한 결과입니다.

## 논점 3: 효율성과 확장성

AI는 수천 명의 지원자를 동시에 평가하며 24시간 운영됩니다. 유니레버는 AI 도입 후 채용 기간을 4개월에서 2주로 단축하고, 면접 대상자를 4배 확대했습니다. 이는 우수 인재 발굴 기회를 민주화하며, 기업은 비용을 70% 절감했습니다. 인간은 피로도와 시간 제약으로 소수만 심층 평가하지만, AI는 모든 지원자에게 공평한 검토 기회를 제공합니다.

[CX_1AC_Q] 부정측 — 교차조사 질문
## [CX_1AC_Q] 부정측 — 교차조사 질문

### 질문 1: AI의 편향성 학습 문제
AI는 과거 데이터로 학습하는데, 데이터 자체가 편향되어 있다면 AI는 그 편향을 재생산하지 않습니까? 아마존이 2018년 AI 채용 시스템을 폐기한 이유가 여성 지원자 차별 때문인데, 이것이 어떻게 "편향 없는" 시스템입니까?

### 질문 2: 예측 정확도의 측정 기준
83% 예측 정확도의 '성과'는 

## 3. 일괄 실행 (run_debate)

전체 토론을 한 번에 실행하고 `DebateResult`로 받습니다.

In [ ]:
result = run_debate("인공지능 개발에 대한 국제적 규제가 필요하다")

print(f"논제: {result.proposition}")
print(f"총 발언 수: {len(result.transcript)}")
print(f"\n{'='*60}")
print("전체 토론 기록")
print(f"{'='*60}")
print(result.format_transcript())
print(f"\n{'='*60}")
print("최종 판정")
print(f"{'='*60}")
print(result.verdict)

## 4. 측별 모델 지정 (모델 대결)

긍정측과 부정측에 서로 다른 LLM을 사용할 수 있습니다.

In [ ]:
# 예시: 같은 프로바이더 내에서 모델만 변경
# OpenAI API 키가 있는 경우 아래 주석을 해제하세요

# result = run_debate(
#     "인공지능 개발에 대한 국제적 규제가 필요하다",
#     aff_provider_name="anthropic",
#     aff_model_name="claude-sonnet-4-5-20250929",
#     neg_provider_name="openai",
#     neg_model_name="gpt-4o",
# )
# print(result.format_transcript())

## 5. 도구 포함 토론

토론 에이전트에게 웹 검색 등 도구를 제공할 수 있습니다.

In [ ]:
# 도구 포함 토론 (TAVILY_API_KEY 필요)

# from Agent_Structure.tools import tool_registry
# search_tools = tool_registry.get_by_tag("search")
# print(f"사용 가능한 검색 도구: {[t.name for t in search_tools]}")

# result = run_debate(
#     "원자력 발전소를 확대해야 한다",
#     tools=search_tools,
# )
# print(result.format_transcript())

## 6. 그래프 구조 확인

In [ ]:
graph, initial_state = create_debate("테스트 논제")

print("초기 상태:")
for k, v in initial_state.items():
    if k == 'round_sequence':
        print(f"  {k}: {len(v)}개 라운드")
    else:
        print(f"  {k}: {repr(v)[:80]}")

print(f"\n그래프 노드: {graph.get_graph().nodes}")